# Memoria Viva — restauración en Google Colab

Este notebook prepara el proyecto, instala las dependencias sin reemplazar PyTorch/CUDA de Colab, carga los modelos en GPU y publica la interfaz Gradio. Elegí **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** si está disponible. Colab asigna hardware según disponibilidad; el notebook se detiene con un mensaje si el runtime no tiene CUDA.

**Entrada del proyecto:** subí desde tu computadora `memoria_viva_colab.zip`, o descargá el ZIP del repositorio desde GitHub con **Code → Download ZIP** y subilo cuando la celda lo solicite. El disco de la VM de Colab es temporal.

El modelo genera píxeles nuevos al completar daños y estima los colores; no recupera evidencia histórica. 


In [1]:
# Diagnóstico de hardware y entorno
import os, torch
print('Python:', __import__('sys').version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
else:
    raise RuntimeError('No hay GPU CUDA. En Colab seleccioná una T4 GPU y volvé a ejecutar este notebook.')

Python: 3.13.16
PyTorch: 2.14.1+cu130
CUDA disponible: True
GPU: Tesla T4
VRAM (GiB): 14.6


In [2]:
# Subí el ZIP del proyecto descargado desde GitHub.
from pathlib import Path
from google.colab import files
import os
import zipfile

subidos = files.upload()
zips = [nombre for nombre in subidos if nombre.lower().endswith('.zip')]
if not zips:
    raise FileNotFoundError('Subí el ZIP del proyecto descargado desde GitHub.')

destino = Path('/content/memoria_viva')
destino.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zips[0]) as archivo_zip:
    archivo_zip.extractall(destino)

# Acepta el ZIP del proyecto o el ZIP de GitHub, que agrega una carpeta contenedora.
if (destino / 'src/gradio_app.py').exists():
    PROYECTO = destino
else:
    candidatos = [
        carpeta for carpeta in destino.iterdir()
        if carpeta.is_dir() and (carpeta / 'src/gradio_app.py').exists()
    ]
    if not candidatos:
        raise FileNotFoundError('El ZIP no contiene src/gradio_app.py. Descargá el ZIP del proyecto Memoria Viva.')
    PROYECTO = candidatos[0]

assert (PROYECTO / 'assets').exists(), 'No se encontró assets en el proyecto.'
os.environ['MEMORIA_VIVA_ROOT'] = str(PROYECTO)


Saving memoria_viva_colab.zip to memoria_viva_colab.zip


In [3]:
# Instalar librerías de la app. Se conserva el PyTorch con CUDA provisto por Colab.
%pip install -q -r "{PROYECTO / 'requirements-colab.txt'}"

In [4]:
# Comprobar imágenes y cargar la app. Los pesos se descargan de Hugging Face
# en la primera inferencia; puede tardar varios minutos.
import sys
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))
from src.gradio_app import crear_app
demo = crear_app()
print('Interfaz preparada. La primera inferencia cargará difusión y RT-DETR en la GPU.')

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Interfaz preparada. La primera inferencia cargará difusión y RT-DETR en la GPU.


In [ ]:
# Iniciar la app y obtener un enlace temporal de Gradio.
# Mantener esta celda ejecutándose durante la demo.
demo.queue().launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9106b9a6dc898471fa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

... (8 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/764 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

  0%|          | 0/18 [00:00<?, ?it/s]

## Flujo de uso
1. Cargá un ejemplo del proyecto o subí una foto propia.
2. Para reparar, pintá encima de rayones/manchas; el área pintada es la máscara de inpainting.
3. Activá reparación, colorización o ambas.
4. Revisá original, reparación, resultado, máscara, mapa de diferencias y análisis RT-DETR.
5. Descargá el resultado. Para demostrar límites, probá una máscara sobre un rostro o una zona sin información suficiente y comentá qué inventó o alteró el modelo.

Para detener la app, interrumpí la celda de lanzamiento. Los enlaces `share=True` son temporales mientras la sesión siga activa; no guardes fotos privadas en una demo pública.